## EJERCICIO 2.7

Modifica el archivo water_data_consumer_pr_2.ipynb para tener consumidores con un group_id común. Cada sensor_id debe producir un plot con los datos producidos por ese sensor_id. Crea una copia de water_data_consumer_pr_2.ipynb modificado y adjunta captura de los dos notebooks modificados ejecutándose a la vez y gestionando diferentes sensor_id y los plots resultantes (1.25 punto).

In [ ]:
import json
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
from kafka import KafkaConsumer
from collections import defaultdict
import os

PLOT_DIR = "sensor_plots"
os.makedirs(PLOT_DIR, exist_ok=True)

MAX_POINTS = 100 

sensor_store = defaultdict(lambda: {
    "timestamps":        [],
    "water_temperature": [],
    "ph_level":          [],
    "turbidity":         [],
    "dissolved_oxygen":  [],
})


def initialize_consumer():
    kafka_topic             = "water_quality_final"
    kafka_bootstrap_servers = ["localhost:9092"]

    consumer = KafkaConsumer(
        kafka_topic,
        bootstrap_servers=kafka_bootstrap_servers,
        value_deserializer=lambda m: json.loads(m.decode("utf-8")),
        auto_offset_reset="latest",
        enable_auto_commit=True,
        group_id="water_quality_processors",
    )
    return consumer


def plot_sensor(sensor_id, data):
    """Save a 2x2 figure with the four water-quality metrics for sensor_id."""
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    fig.suptitle(f"Sensor ID: {sensor_id}", fontsize=14, fontweight="bold")

    axes[0, 0].plot(data["timestamps"], data["water_temperature"], color="blue")
    axes[0, 0].set_title("Water Temperature")
    axes[0, 0].set_ylabel("°C")
    axes[0, 0].tick_params(axis="x", rotation=45)

    axes[0, 1].plot(data["timestamps"], data["ph_level"],          color="green")
    axes[0, 1].set_title("pH Level")
    axes[0, 1].set_ylabel("pH")
    axes[0, 1].tick_params(axis="x", rotation=45)

    axes[1, 0].plot(data["timestamps"], data["turbidity"],         color="orange")
    axes[1, 0].set_title("Turbidity")
    axes[1, 0].set_ylabel("NTU")
    axes[1, 0].tick_params(axis="x", rotation=45)

    axes[1, 1].plot(data["timestamps"], data["dissolved_oxygen"],  color="red")
    axes[1, 1].set_title("Dissolved Oxygen")
    axes[1, 1].set_ylabel("mg/L")
    axes[1, 1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    out_path = os.path.join(PLOT_DIR, f"water_quality_{sensor_id}.png")
    plt.savefig(out_path)
    plt.close(fig)


def update_plot(consumer):
    try:
        for message in consumer:
            sd = message.value  
            print(f"Received: {sd}")

            sid  = sd["sensor_id"]
            data = sensor_store[sid]

            data["timestamps"].append(sd["timestamp"])
            data["water_temperature"].append(sd["water_temperature"])
            data["ph_level"].append(sd["ph_level"])
            data["turbidity"].append(sd["turbidity"])
            data["dissolved_oxygen"].append(sd["dissolved_oxygen"])

            # Keep only the last MAX_POINTS readings
            for key in data:
                if len(data[key]) > MAX_POINTS:
                    data[key].pop(0)

            plot_sensor(sid, data)
            break  

    except KeyboardInterrupt:
        print("Stopped consuming messages.")
        consumer.close()


consumer = initialize_consumer()
print("Subscribed to Kafka topic 'water_quality' (group_id='water_quality_processors').")

try:
    while True:
        update_plot(consumer)
except KeyboardInterrupt:
    print("Stopped visualization.")
    consumer.close()

Subscribed to Kafka topic 'water_quality' (group_id='water_quality_processors').
Received: {'sensor_id': 'sensor_000', 'timestamp': 1772386599, 'water_temperature': 28.95, 'ph_level': 7.88, 'turbidity': 41.97, 'dissolved_oxygen': 11.53}
Received: {'sensor_id': 'sensor_002', 'timestamp': 1772386599, 'water_temperature': 29.48, 'ph_level': 8.01, 'turbidity': 25.44, 'dissolved_oxygen': 8.84}
Received: {'sensor_id': 'sensor_005', 'timestamp': 1772386599, 'water_temperature': 29.67, 'ph_level': 8.26, 'turbidity': 35.94, 'dissolved_oxygen': 6.65}
Received: {'sensor_id': 'sensor_006', 'timestamp': 1772386599, 'water_temperature': 28.77, 'ph_level': 7.97, 'turbidity': 19.15, 'dissolved_oxygen': 11.67}
Received: {'sensor_id': 'sensor_003', 'timestamp': 1772386599, 'water_temperature': 29.34, 'ph_level': 8.15, 'turbidity': 14.39, 'dissolved_oxygen': 9.01}
Received: {'sensor_id': 'sensor_004', 'timestamp': 1772386599, 'water_temperature': 29.97, 'ph_level': 8.75, 'turbidity': 45.56, 'dissolved_ox